# This notebook intends to address task3 -- Fractal Capacity Planning

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus

## Demand Calculation

In [2]:
# parameter settings:

products = ['A1', 'A2', 'A3', 'B1', 'B2']

# weekly demand 
demand_data = {
    'Weekly_Mean': [
        962,  
        1923, 
        2500, 
        1154, 
        1538  
    ],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18] 
}

demand_df = pd.DataFrame(demand_data, index=products) 

# the BOM of requirements of each Products (A, B) from each Parts (P1~P20)
bom_df = pd.read_csv('../Data/BOM.csv', index_col=0).T # for the later matrix multiplication dim align
print('BOM of product to parts:\n', bom_df.head())


BOM of product to parts:
     A1  A2  A3  B1  B2
P1   1   2   4   4   1
P2   4   0   2   2   0
P3   0   0   0   4   1
P4   0   1   2   2   0
P5   0   1   2   0   0


In [3]:
# calculate the target capacity for products
Z_score = norm.ppf(0.995)

demand_df['Weekly_std'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']
# round up to maintain integer
demand_df['Product_Capacity'] = np.ceil(demand_df['Weekly_Mean'] + demand_df['Weekly_std']*Z_score)


# Aggregate to parts demand
part_demand = pd.DataFrame()
part_demand['Part_Weekly_Capacity'] = bom_df.dot(demand_df['Product_Capacity'])

part_demand['Part_Weekly_Capacity'] = np.ceil(part_demand['Part_Weekly_Capacity']) 

# Fix: Strip whitespace from the index
part_demand.index = part_demand.index.str.strip()

part_demand = part_demand / 4

print('weekly capacity for each product:\n', demand_df['Product_Capacity'])
print('\nweekly capacity for each part:\n', part_demand)

weekly capacity for each product:
 A1    1334.0
A2    2914.0
A3    3788.0
B1    1511.0
B2    2252.0
Name: Product_Capacity, dtype: float64

weekly capacity for each part:
      Part_Weekly_Capacity
P1                7652.50
P2                3983.50
P3                2074.00
P4                3378.00
P5                2622.50
P6                1459.50
P7                5166.00
P8                1511.00
P9                3351.00
P10               4572.25
P11               2919.00
P12               3586.00
P13               1793.00
P14               6689.50
P15                755.50
P16               4516.50
P17               1793.00
P18               4314.50
P19               7319.00
P20               5431.25


## Capacity Planning

after validating the demand for each part, in this case each parts are produced through a series of processes naming A, B, C...M. Meanwhile, some machine can do one step and some can do multiple and that is showcased via their naming (e.g. machien AB can do A and B step)

Thus we formulte the problem as followsings:

### Sets
- $P$: set of products (parts), index $p$
- $S$: set of process steps, index $s$
- $M$: set of machine types (or pools), index $m$

### Parameters

$D_{ps} \ge 0$: required processing units for product $p$ at step $s$ in the planning period

$\eta_{ps} \ge 0$: productivity of machines on $Step_ps$, while machines do not casue differences in time of processing, there is no $m$ dimension

$$\eta_{ps} = \frac{60}{t_{ps}}$$

$A_m > 0$: available hours per installed machine of type $m$ in the period (calendar hours $\times$ planned shifts)

$\mathrm{OEE}_m \in (0,1]$: effective efficiency factor (availability $\times$ performance $\times$ quality) for type $m$

$K_m \ge 0$: installation/fixed cost per machine of type $m$ 

$C^{\text{lab}}_m \ge 0$: labor cost per machine-hour used on type $m$ --  __Not considered anymore__

$E_{ms} \in \{0,1\}$: eligibility indicator; $E_{ms} = 1$ if type $m$ can perform step $s$, else $0$

for efficiency, we define effective per-machine hours:
$$A_m^{eff} = A_m * \mathrm{OEE}$$

### Decision Vriables
- $y_m \in \mathbb{Z}_{+}$: number of machines of type $m$ to install/plan
- $x_{mps} \ge 0$: machine-hours of type $m$ allocated to product $p$ at step $s$

### Objective
$$\min Z = \sum_{m \in M} K_m y_m $$

### Constraints

- Unite demand based satisfaction of each step:
$$\sum_m \eta_{ps} * X_{mps} \ge D_{ps} \quad \forall p \in P, s \in S$$

- Capacity coupling by machine type:
$$\sum_{p \in P} \sum_{s \in S} x_{mps} \leq A_m^{\text{eff}} y_m \quad \forall m \in M$$

- Eligibility (step-machine):
$$x_{mps} = 0 \quad \text{whenever } E_{ms} = 0$$

- Variable domains:
$$x_{mps} \geq 0 \quad \forall m, p, s; \qquad y_m \in \mathbb{Z}_+ \quad \forall m$$

__Note that the horizon of timeframe of all the numerical values are within a wek__

In [4]:
# load parameters

path = "../Data"  # Go up one level then into Data
step = pd.read_csv(f"{path}/Steps.csv", index_col=0)
timecost = pd.read_csv(f"{path}/Timecost.csv", index_col=0)
 
print(step.head())
print("\n\n", timecost.head())

     Step 1 Step 2 Step 3 Step 4 Step 5 Step 6 Step 7
Part                                                 
P1        B      A      B      C      D      I      J
P2        A      C      D      H      J    NaN    NaN
P3        B      D      C      I      J    NaN    NaN
P4        A      B      D      G      H    NaN    NaN
P5        B      C      C      I    NaN    NaN    NaN


       Step 1  Step 2  Step 3  Step 4  Step 5  Step 6  Step 7
Part                                                        
P1      2.50    1.00    2.50    0.50    2.50    1.25     2.5
P2      1.25    0.50    2.50    1.00    2.50     NaN     NaN
P3      1.75    3.00    0.75    1.50    2.50     NaN     NaN
P4      1.00    2.00    3.00    0.25    1.25     NaN     NaN
P5      1.50    0.75    3.50    1.75     NaN     NaN     NaN


In [5]:
# calculate the productivity matrix

productivity = 60 / timecost

### calculate the effective working time
- 90% efficiency and 98% reliability of all equipment, with 100% quality at each operation
 
- all the machines are working one or two 8hr shift perday and 5 days a week 

Here we assume one shifts per day for larger tolerance in the planning. However, this can be changed

In [6]:
OEE = 0.9 * 0.98
A = 5 * 8 
A_eff = A * OEE

In [7]:
print(A_eff)

35.28


In [ ]:
# load the parameters
machine = pd.read_csv("../Data/Equipment.csv", index_col='Equipment', usecols=['Equipment', 'Installed price', 'C1', 'C2', 'C3'])

K_m = machine.loc[:, 'Installed price']

## Solver Calling

In [ ]:
from case03.Task3.Solver import CapacityPlanningSolver

# transform data structure first

part_demand_dict = part_demand['Part_Weekly_Capacity'].to_dict()
K_m_dict= K_m.to_dict()
# C_m_lab_dict = labor_cost['Salary'].to_dict()


# call solver class
solver = CapacityPlanningSolver()

# create eligibility matrix
all_steps = solver._extract_all_steps(step)
eligibility = {}
for machine in machine.index:
    eligibility[machine] = {}
    for s in all_steps:
        # Machine 'AB' can do steps 'A' and 'B'
        # Machine 'ABCD' can do steps 'A', 'B', 'C', 'D'
        eligibility[machine][s] = 1 if s in machine else 0

In [ ]:
# run solver
results = solver.solve(
    part_demand=part_demand_dict,
    step_df=step,
    productivity_df=productivity,
    K_m=K_m,
    A_eff=A_eff,
    eligibility=eligibility,
    verbose=True
)


CAPACITY PLANNING OPTIMIZATION

Building optimization model...
Model Statistics:
  - Variables: 6786
  - Constraints: 5990
  - Demand-step combinations: 84

Solving MIP model...
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/tianyihu/.venvs/general/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/1q/yvvtgq1134x569hhykb6ll100000gn/T/c2356af1a97242b1a2b353be1db23eba-pulp.mps -sec 300 -timeMode elapsed -branch -printingOptions all -solution /var/folders/1q/yvvtgq1134x569hhykb6ll100000gn/T/c2356af1a97242b1a2b353be1db23eba-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 5995 COLUMNS
At line 18994 RHS
At line 24985 BOUNDS
At line 25012 ENDATA
Problem MODEL has 5990 rows, 6786 columns and 12920 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 300
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1.0149e+08 - 0.00 seconds

In [11]:
# get the distribution of time 

print(f"The machine allocations:", solver.get_machine_allocation())


The machine allocations:    Machine  Quantity  Installation_Cost  Utilization_%   Hours_Used  \
0        A         8          1200000.0     100.000001   282.240003   
1      ABC         1           300000.0      86.985071    30.688333   
2        B        16          3200000.0      99.999999   564.479997   
3        C         6          1500000.0     100.000000   211.679999   
4       CD         1           350000.0     100.000000    35.280000   
5        D        23          6900000.0     100.000000   811.440000   
6        E        16          6400000.0     100.000001   564.480004   
7       EG         1           450000.0      77.631332    27.388334   
8        F        16          6400000.0      98.808675   557.755207   
9        G        13          5200000.0      99.999999   458.639996   
10       H        25         25000000.0      99.114466   874.189588   
11       I        26         13000000.0      99.275961   910.638538   
12       J        39         29250000.0      98.6096

In [ ]:
# task1_result = solver.save_results(filepath='Task1_CapacityPlanning_Results.csv')

Results saved to: Task1_CapacityPlanning_Results.csv
